# Homo-filter diffusion visualization



In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy.linalg import eigh
import string


def load_homogeneous_similarity_graph(csv_path, knn_k=10):
    """
    读取 ncRNA 相似性矩阵，并构造用于扩散的同构图邻接矩阵。
    """
    df = pd.read_csv(csv_path, index_col=0)

    if df.shape[0] != df.shape[1]:
        raise ValueError(
            f"相似性矩阵必须是方阵，但当前形状为 {df.shape}。"
        )

    A = df.values.astype(np.float64)
    node_names = df.index.astype(str).tolist()

    # 数值稳定性处理
    A = np.nan_to_num(A, nan=0.0, posinf=0.0, neginf=0.0)

    # 保证无向同构图：相似性矩阵对称化
    A = 0.5 * (A + A.T)

    # 去除自环。热核扩散中不需要保留相似度对角线上的 1
    np.fill_diagonal(A, 0.0)

    # 相似性图通常很稠密；保留 Top-k 邻居以形成局部相似性图
    if knn_k is not None:
        A = sparsify_similarity_knn(A, k=knn_k)

    return A, node_names


def sparsify_similarity_knn(A, k=10):
    """
    对相似性矩阵进行 Top-k 邻居稀疏化。

    对每个节点仅保留相似度最高的 k 个邻居，
    再通过 max(A, A^T) 恢复为无向图。
    """
    A = np.asarray(A, dtype=np.float64).copy()
    n = A.shape[0]

    if k <= 0 or k >= n:
        return A

    np.fill_diagonal(A, 0.0)

    mask = np.zeros_like(A, dtype=bool)

    for i in range(n):
        top_idx = np.argsort(A[i])[::-1][:k]
        mask[i, top_idx] = True

    A_knn = np.where(mask, A, 0.0)

    # 若 i 选择 j 或 j 选择 i，则保留该边
    A_knn = np.maximum(A_knn, A_knn.T)

    np.fill_diagonal(A_knn, 0.0)
    return A_knn


def normalized_laplacian(A, eps=1e-12):
    """
    归一化图拉普拉斯矩阵：

        L = I - D^(-1/2) A D^(-1/2)
    """
    A = np.asarray(A, dtype=np.float64)

    deg = A.sum(axis=1)
    deg_inv_sqrt = 1.0 / np.sqrt(deg + eps)

    A_norm = deg_inv_sqrt[:, None] * A * deg_inv_sqrt[None, :]
    L = np.eye(A.shape[0], dtype=np.float64) - A_norm

    return L


def laplacian_eig(A):
    """
    拉普拉斯谱分解：

        L = U Lambda U^T
    """
    L = normalized_laplacian(A)
    lambdas, U = eigh(L)

    # 归一化拉普拉斯理论特征值范围为 [0, 2]
    lambdas = np.clip(lambdas, 0.0, 2.0)

    return lambdas, U


def wavelet_basis_impulse_response(U, lambdas, center_node, s):
    """
    GWNN 中定义的小波基响应：

        psi_s delta_c = exp(sL) delta_c
    """
    delta_hat = U[center_node, :]
    wavelet_filter = np.exp(s * lambdas)

    psi_delta = U @ (wavelet_filter * delta_hat)

    return psi_delta


def prepare_response_for_display(
    response,
    center_node=None,
    suppress_center=True,
    source_upper=None,
    display_upper=5.0,
    gamma=0.55,
    boost_above_mean=True,
    threshold_ratio=0.5,
    high_gamma=0.40,
    eps=1e-12
):
    """
    将原始小波响应映射到 [0, display_upper]，用于可视化。

    gamma:
        全局增强弱响应。gamma < 1 时，弱响应更明显。

    boost_above_mean:
        是否额外增强色带平均值以上的响应。

    threshold_ratio:
        阈值占色带范围的比例。
        默认 0.5，即色带平均值 display_upper / 2。

    high_gamma:
        对超过阈值部分的增强强度。
        取值越小，超过平均值的点越突出。
        建议 0.35 ~ 0.65。
    """
    if not (0 < display_upper <= 5.0):
        raise ValueError(
            f"display_upper 必须位于 (0, 5]，当前为 {display_upper}"
        )

    if not (0 < threshold_ratio < 1):
        raise ValueError("threshold_ratio 必须位于 (0, 1)")

    if high_gamma <= 0:
        raise ValueError("high_gamma 必须大于 0")

    # 用幅值作为绘图响应强度
    r = np.abs(np.asarray(response, dtype=np.float64).copy())

    if suppress_center and center_node is not None:
        r[center_node] = 0.0

    # 每个尺度自身的原始截断上限
    if source_upper is None:
        source_upper = np.percentile(r, 99.0)

    source_upper = max(float(source_upper), eps)

    # 映射到 [0, 1]
    x = np.clip(r, 0.0, source_upper) / source_upper

    # 全局弱响应增强
    x = x ** gamma

    # 额外增强色带平均值以上的节点
    if boost_above_mean:
        threshold = threshold_ratio
        mask = x > threshold

        z = (x[mask] - threshold) / (1.0 - threshold)

        # 保持 threshold 不变，保持最大值仍为 1
        x[mask] = threshold + (1.0 - threshold) * (z ** high_gamma)

    # 映射至该尺度自己的显示范围 [0, display_upper]
    return x * display_upper


def adj_to_networkx_graph(A):
    """
    邻接矩阵转为 NetworkX 无向加权图。
    """
    A = np.asarray(A, dtype=np.float64)

    G = nx.Graph()
    num_nodes = A.shape[0]
    G.add_nodes_from(range(num_nodes))

    rows, cols = np.nonzero(np.triu(A, k=1))

    for i, j in zip(rows, cols):
        G.add_edge(i, j, weight=A[i, j])

    return G


def choose_subgraph_by_response(
    G,
    response_list,
    center_node,
    top_k=150,
    add_neighbors=True
):
    """
    根据多个尺度的响应联合选择子图节点。

    对每个节点取各尺度响应中的最大值，选择 Top-k 节点，
    从而保证不同尺度下的重要节点都可能被展示。
    """
    response_stack = np.stack(response_list, axis=0)
    combined_response = np.abs(response_stack).max(axis=0)

    selected = set(np.argsort(-combined_response)[:top_k].tolist())
    selected.add(center_node)

    if add_neighbors:
        selected.update(list(G.neighbors(center_node)))

    selected = sorted(selected)
    G_sub = G.subgraph(selected).copy()

    return G_sub


def get_layout(G, layout="spring", seed=42):
    """
    计算所有尺度共享的图布局。
    """
    if layout == "spring":
        return nx.spring_layout(
            G,
            seed=seed,
            iterations=200,
            k=0.5
        )

    if layout == "kamada_kawai":
        return nx.kamada_kawai_layout(G)

    raise ValueError("layout must be 'spring' or 'kamada_kawai'")

# cmap="viridis",cmap="jet",cmap="coolwarm",
def draw_diffusion_panel(
    ax,
    G,
    pos,
    response_show,
    center_node,
    title,
    vmax,
    cmap="coolwarm",
    node_size_base=12,
    node_size_scale=80,
    edge_alpha=0.12,
    edge_width=0.35
):
    nodes = list(G.nodes())

    node_colors = np.array([response_show[n] for n in nodes])

    # 节点大小单独缩放，避免 exp(tL) 数值过大导致节点异常
    size_values = np.clip(node_colors / max(vmax, 1e-12), 0.0, 1.0)
    node_sizes = node_size_base + node_size_scale * size_values

    nx.draw_networkx_edges(
        G,
        pos,
        ax=ax,
        edge_color="lightgray",
        alpha=edge_alpha,
        width=edge_width
    )

    sc = nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax,
        nodelist=nodes,
        node_color=node_colors,
        node_size=node_sizes,
        cmap=cmap,
        vmin=0.0,
        vmax=vmax,
        linewidths=0.0
    )

    if center_node in G.nodes:
        nx.draw_networkx_nodes(
            G,
            pos,
            ax=ax,
            nodelist=[center_node],
            node_color="red",
            node_shape="*",
            node_size=260,
            edgecolors="black",
            linewidths=0.6
        )

    ax.set_title(title, fontsize=12)
    ax.set_axis_off()

    return sc


def plot_single_scale_heat_diffusions(
    A,
    scales=(1.0, 4.0, 8.0),
    center_node=0,
    center_name=None,
    use_subgraph=True,
    top_k=150,
    layout="spring",
    seed=42,
    clip_percentile=99.0,
    gamma=0.35,
    ncols=3,
    vmax_by_scale=None,
    display_vmax_by_scale=None,
    source_upper_by_scale=None,
    enhancement_by_scale=None,
    save_path=None
):
    """
    在 ncRNA 同构图上绘制多个单尺度热核扩散结果。

    每一幅子图对应一个独立尺度：

        h_t = exp(-tL) delta_c

    """
    A = np.asarray(A, dtype=np.float64)
    scales = list(scales)

    if enhancement_by_scale is None:
        enhancement_by_scale = {}

    if display_vmax_by_scale is None:
        display_vmax_by_scale = {
            t: 5.0 for t in scales
        }

    for t, vmax in display_vmax_by_scale.items():
        if not (0 < vmax <= 5.0):
            raise ValueError(
                f"尺度 t={t} 的显示上限必须位于 (0, 5]，当前为 {vmax}"
            )

    if center_node < 0 or center_node >= A.shape[0]:
        raise ValueError(
            f"center_node={center_node} 超出节点索引范围 [0, {A.shape[0] - 1}]。"
        )

    lambdas, U = laplacian_eig(A)

    # 分别计算各单尺度热核扩散
    responses_raw = []
    responses_show = []

    for t in scales:
        h_t = wavelet_basis_impulse_response(
            U=U,
            lambdas=lambdas,
            center_node=center_node,
            s=t
        )

        # 原始显示截断上限：未指定时自动取该尺度 99% 分位数
        source_upper = None
        if source_upper_by_scale is not None:
            source_upper = source_upper_by_scale.get(t, None)

        # 色带显示上限：由你控制，最大不超过 5
        display_upper = display_vmax_by_scale[t]

        # 读取当前尺度自己的可视化增强参数
        enhance_cfg = enhancement_by_scale.get(t, {})

        boost_above_mean = enhance_cfg.get("boost_above_mean", True)
        threshold_ratio = enhance_cfg.get("threshold_ratio", 0.3)
        high_gamma = enhance_cfg.get("high_gamma", 0.3)

        h_t_show = prepare_response_for_display(
            response=h_t,
            center_node=center_node,
            suppress_center=True,
            source_upper=source_upper,
            display_upper=display_upper,
            gamma=gamma,

            boost_above_mean=boost_above_mean,
            threshold_ratio=threshold_ratio,
            high_gamma=high_gamma
        )



        responses_raw.append(h_t)
        responses_show.append(h_t_show)

    # 若没有人工指定，则每个尺度自动取自身 99% 分位数作为上限
    if vmax_by_scale is None:
        vmax_by_scale = {
            t: np.percentile(resp, 99.0)
            for t, resp in zip(scales, responses_show)
        }

    # 避免上限为 0
    vmax_by_scale = {
        t: max(v, 1e-12)
        for t, v in vmax_by_scale.items()
    }

    print("\nColor ranges for each scale:")
    for t in scales:
        print(f"t={t}: [0, {vmax_by_scale[t]:.4e}]")


    G = adj_to_networkx_graph(A)

    # 所有尺度共享同一个子图和布局，便于横向比较
    if use_subgraph:
        G_plot = choose_subgraph_by_response(
            G=G,
            response_list=responses_show,
            center_node=center_node,
            top_k=top_k,
            add_neighbors=True
        )
    else:
        G_plot = G

    pos = get_layout(G_plot, layout=layout, seed=seed)

    n_panels = len(scales)
    nrows = math.ceil(n_panels / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.8 * ncols, 4.2 * nrows)
    )

    axes = np.array(axes).reshape(-1)

    panel_labels = list(string.ascii_lowercase)
    image_handle = None

    for idx, t in enumerate(scales):
        vmax = display_vmax_by_scale[t]

        image_handle = draw_diffusion_panel(
            ax=axes[idx],
            G=G_plot,
            pos=pos,
            response_show=responses_show[idx],
            center_node=center_node,
            title=f"({panel_labels[idx]}) $t={t}$",
            vmax=vmax
        )

        cbar = fig.colorbar(
            image_handle,
            ax=axes[idx],
            fraction=0.046,
            pad=0.03
        )
        cbar.set_label("Wavelet Response", fontsize=12)
        cbar.ax.tick_params(labelsize=10)

    # 删除未使用的子图
    for idx in range(n_panels, len(axes)):
        axes[idx].axis("off")

    fig.suptitle(
        f"Single-scale Wavelet Basis on ncRNA Homo-Graph",
        fontsize=14,
        y=0.98
    )

    fig.subplots_adjust(
        left=0.02,
        right=0.98,
        bottom=0.04,
        top=0.86,
        wspace=0.28,
        hspace=0.12
    )

    if save_path is not None:
        save_dir = os.path.dirname(save_path)
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)

        plt.savefig(save_path, dpi=600, bbox_inches="tight")

        if save_path.lower().endswith(".png"):
            pdf_path = save_path[:-4] + ".pdf"
            svg_path = save_path[:-4] + ".svg"
        else:
            pdf_path = save_path + ".pdf"
            svg_path = save_path + ".svg"

        plt.savefig(pdf_path, bbox_inches="tight")
        plt.savefig(svg_path, bbox_inches="tight")

    plt.show()

    return {
        "lambdas": lambdas,
        "responses_raw": dict(zip(scales, responses_raw)),
        "responses_show": dict(zip(scales, responses_show)),
        "graph": G_plot,
        "positions": pos
    }


if __name__ == "__main__":

    # 你的 ncRNA 同构相似性图
    csv_path = r".\ncRNA_sim_ncDR.csv"

    # 每个节点只保留最相似的 k 个邻居。
    A_rna, node_names = load_homogeneous_similarity_graph(
        csv_path=csv_path,
        knn_k=15
    )

    print("ncRNA homogeneous graph shape:", A_rna.shape)
    print("Number of ncRNA nodes:", len(node_names))
    print("Center node:", node_names[0])

    # 选择中心节点：
    # 可改为任意 ncRNA 的索引，例如 center_node = 20
    center_node = 0

    results = plot_single_scale_heat_diffusions(
        A=A_rna,
        scales=( 1.0, 2.0, 3.0),
        center_node=center_node,
        center_name=node_names[center_node],
        use_subgraph=True,
        top_k=250,
        layout="spring",
        seed=42,
        clip_percentile=99.0,
        gamma=0.35,
        ncols=3,
        display_vmax_by_scale={
        1.0: 1,
        2.0: 1,
        3.0: 1
        },
        enhancement_by_scale={
            # t=1：只突出最强的一小部分节点
            1.0: {
                "threshold_ratio": 0.55,
                "high_gamma": 0.45
            },

            # t=2：从 40% 色带位置开始强化
            2.0: {
                "threshold_ratio": 0.40,
                "high_gamma": 0.30
            },

            # t=3：更多节点进入强化区，高响应更快变黄/红
            3.0: {
                "threshold_ratio": 0.30,
                "high_gamma": 0.22
            }
        },
        save_path="Homo_filter_diffusion_restrat_noNormalization3/ncRNA_single_scale_heat_diffusion_123.png"
    )


    '''
    threshold_ratio 越小：越多节点会被纳入“高响应增强”区域。
    high_gamma 越小：超过阈值后的节点越快接近色带上限，黄色/红色更明显
    threshold_ratio=0.25：从色带四分之一位置开始增强
    '''